# Config

In [236]:
MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green
MAX_HANDS = 2
CAMERA = 0
FPS = 30
REC_TEST = False
frame_idx = 0

# Load mediapipe Hand Landmarks and Opencv2

In [237]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Mediapipe Utils

In [238]:
def create_hand_detector(model_path: str = "hand_landmarker.task", num_hands: int = 1):
    """
    MediaPipe HandLandmarker 생성.
    """
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=num_hands,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
    )
    detector = vision.HandLandmarker.create_from_options(options)
    return detector

def get_drawing_utils():
    """
    랜드마크 시각화를 위한 연결 정보 반환.
    """
    return vision.HandLandmarksConnections.HAND_CONNECTIONS

def capture_frame(cap: cv2.VideoCapture):
    """
    웹캠에서 프레임 1장을 읽어 반환.
    실패 시 (False, None) 반환.
    """
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    if not ret:
        return False, None
    return True, frame

def convert_bgr_to_mp_image(bgr_frame: np.ndarray):
    """
    OpenCV BGR 이미지를 RGB 및 MediaPipe Image로 변환.
    """
    rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return rgb_frame, mp_image

def detect_hand_landmarks(detector, bgr_frame: np.ndarray):
    """
    BGR 프레임을 받아 MediaPipe HandLandmarker로 손 랜드마크 검출.
    반환:
      - rgb_frame
      - detection_result
    """
    global frame_idx
    rgb_frame, mp_image = convert_bgr_to_mp_image(bgr_frame)
    timestamp_ms = int(frame_idx * 1000 / FPS)
    # detection_result = detector.detect(mp_image)
    detection_result = detector.detect_for_video(mp_image, timestamp_ms)
    frame_idx += 1


    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    for idx in range(len(hand_landmarks_list)):
        hand_label = handedness_list[idx][0].category_name

        if hand_label == "Right":
            handedness_list[idx][0].category_name = "Left"
        elif hand_label == "Left":
            handedness_list[idx][0].category_name = "Right"

    return rgb_frame, detection_result

def draw_landmarks_on_image(rgb_image, detection_result, hand_connections):
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(rgb_image)
    image_h, image_w, _ = annotated_image.shape

    for idx, hand_landmarks in enumerate(hand_landmarks_list):
        # 관절 연결선 그리기
        for connection in hand_connections:
            start_idx, end_idx = connection.start, connection.end
            start_lm = hand_landmarks[start_idx]
            end_lm = hand_landmarks[end_idx]
            start_point = (int(start_lm.x * image_w), int(start_lm.y * image_h))
            end_point = (int(end_lm.x * image_w), int(end_lm.y * image_h))
            cv2.line(annotated_image, start_point, end_point, (0, 255, 255), 2)

        # 관절 포인트 그리기
        for lm in hand_landmarks:
            px = int(lm.x * image_w)
            py = int(lm.y * image_h)
            cv2.circle(annotated_image, (px, py), 4, (255, 80, 80), -1)

        if idx < len(handedness_list):
            handedness = handedness_list[idx]
            x_coordinates = [landmark.x for landmark in hand_landmarks]
            y_coordinates = [landmark.y for landmark in hand_landmarks]
            text_x = int(min(x_coordinates) * image_w)
            text_y = int(min(y_coordinates) * image_h) - MARGIN

            cv2.putText(
                annotated_image,
                f"{handedness[0].category_name}",
                (text_x, text_y),
                cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE,
                HANDEDNESS_TEXT_COLOR,
                FONT_THICKNESS,
                cv2.LINE_AA,
            )
    return annotated_image

# Recognition Test

In [239]:
if REC_TEST:
    cap = cv2.VideoCapture(CAMERA)
    if not cap.isOpened():
        raise RuntimeError(f"웹캠을 열 수 없습니다. camera_index={CAMERA}")
    
    detector = create_hand_detector(model_path="hand_landmarker.task", num_hands=MAX_HANDS)
    hand_connections = get_drawing_utils()

    try:
        while True:
            ret, frame = capture_frame(cap)
            if not ret:
                print("프레임을 읽지 못했습니다. 종료합니다.")
                break

            rgb_frame, detection_result = detect_hand_landmarks(detector, frame)
            annotated_image = draw_landmarks_on_image(rgb_frame, detection_result, hand_connections)

            # OpenCV는 BGR 기준으로 표시하므로 RGB -> BGR로 변환
            display_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)
            cv2.imshow("MediaPipe Hand Landmarks", display_image)

            # ESC 또는 q 입력 시 종료
            key = cv2.waitKey(1) & 0xFF
            if key == 27 or key == ord("q"):
                break
    finally:
        cap.release()
        cv2.waitKey(-1)
        cv2.destroyAllWindows()

# Model

In [240]:
CSV_PATH = "data_test_convted.csv"

## KNN

In [241]:
def load_knn():
    import os
    if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) > 0:
        file = np.genfromtxt(CSV_PATH, delimiter=',')
        if file.ndim == 1: file = np.array([file])
        X = file[:, :-1].astype(np.float32)
        y = file[:, -1].astype(np.float32)
        knn = cv2.ml.KNearest_create()
        knn.train(X, cv2.ml.ROW_SAMPLE, y)
    else:
        knn = None
    return knn


def predict_hand_gesture_by_knn(detection_result, knn):

    left_result = None
    right_result = None

    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    if not hand_landmarks_list:
        return {"left": None, "right": None}
    

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        hand_label = handedness_list[idx][0].category_name

        hand_side = -1
        if hand_label == "Right":
            hand_side = 1
        elif hand_label == "Left":
            hand_side = 0
        # hand_side = hand_side * 100

        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

        v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19],:]
        v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],:]

        v = v2 - v1
        v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]

        angle = np.arccos(np.einsum(
            'nt,nt->n',
            v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
            v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]
        ))

        angle = np.degrees(angle)

        v_p1 = joint[[0, 0], :]
        v_p2 = joint[[5, 17], :]
        v_palm = v_p2 - v_p1

        palm_normal = np.cross(v_palm[0], v_palm[1])
        palm_normal = palm_normal / np.linalg.norm(palm_normal)
        palm_normal = palm_normal * 500

        palm_size = np.linalg.norm(joint[17] - joint[5]) + 1e-6

        tdi = np.linalg.norm(joint[4] - joint[8]) / palm_size
        tdm = np.linalg.norm(joint[4] - joint[12]) / palm_size

        full_data = np.append(angle, palm_normal)
        full_data = np.append(full_data, tdi)
        full_data = np.append(full_data, tdm)
        full_data = np.append(full_data, hand_side)

        data = np.array([full_data], dtype=np.float32)
    
        # print(data)
        # TODO Change model
        ret, results, neighbours, dist = knn.findNearest(data, 3)
        # if dist[0][0] < 2000.0:
        label = int(results[0][0])

        if hand_label == "Right":
            right_result = label
        elif hand_label == "Left":
            left_result = label

    return {
        "left": left_result,
        "right": right_result
    }

# DNN

### Config

In [242]:
DATA_HEADER = [
    "th1",
    "th2",
    "th3",
    "in1",
    "in2",
    "in3",
    "mi1",
    "mi2",
    "mi3",
    "ri1",
    "ri2",
    "ri3",
    "pi1",
    "pi2",
    "pi3",
    # TODO Remove
    # "pnx",
    # "pny",
    # "pnz",
    # "thumb_index_tip_dist",
    # "thumb_middle_tip_dist",
    "hand_side",
]


LABEL_HEADER = [
    # finger labels
    # 1 = open, 0 = folded
    "th_open",
    "in_open",
    "mi_open",
    "ri_open",
    "pi_open",
    "ti_touch",
    "tm_touch",
    # "clockwise",
    # "anticlockwise"
]

EPOCH = 100
BATCH_SIZE = 32

CSV_PATHS = ["train_data_left.csv", "train_data_right.csv"]

In [243]:
import pandas as pd
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

def load_dataset(csv_paths, data_header, label_header):
    dfs = []
    for csv_file in csv_paths:
        df = pd.read_csv(csv_file)
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)
    print(df[LABEL_HEADER].describe())

    X = df[data_header].values.astype(np.float32)
    y = df[label_header].values.astype(np.float32)

    return X, y

def prepare_data(X, y):
    scaler = ColumnTransformer(
        transformers=[
            ("scale", StandardScaler(), list(range(X.shape[1]-1)))
        ],
        remainder="passthrough"
    )

    scaler.fit(X)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True
    )


    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)

    return X_train, X_val, y_train, y_val, scaler



### Model

In [244]:
import tensorflow as tf
import pickle


def tristate_from_prob(y_pred, low=0.2, high=0.8):
    """
    반환값:
      0  -> negative
      1  -> positive
     -1  -> uncertain
    """
    neg = tf.cast(y_pred <= low, tf.int32) * 0
    pos = tf.cast(y_pred >= high, tf.int32) * 1

    certain_mask = tf.logical_or(y_pred <= low, y_pred >= high)
    certain_mask = tf.cast(certain_mask, tf.int32)

    tri = tf.where(
        certain_mask == 1,
        pos + neg,  # pos는 1, neg는 0
        tf.fill(tf.shape(tf.cast(y_pred, tf.int32)), -1),
    )
    return tri


class ExactMatchUncertainWrong(tf.keras.metrics.Metric):
    def __init__(self, low=0.2, high=0.8, name="exact_match_wrong", **kwargs):
        super().__init__(name=name, **kwargs)
        self.low = low
        self.high = high
        self.correct = self.add_weight(name="correct", initializer="zeros")
        self.total = self.add_weight(name="total", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.int32)
        tri = tristate_from_prob(y_pred, self.low, self.high)

        # uncertain 포함되면 바로 false
        no_uncertain = tf.reduce_all(tf.not_equal(tri, -1), axis=1)
        all_correct = tf.reduce_all(tf.equal(tri, y_true), axis=1)
        match = tf.logical_and(no_uncertain, all_correct)

        match = tf.cast(match, tf.float32)

        self.correct.assign_add(tf.reduce_sum(match))
        self.total.assign_add(tf.cast(tf.shape(y_true)[0], tf.float32))

    def result(self):
        return tf.math.divide_no_nan(self.correct, self.total)

    def reset_state(self):
        self.correct.assign(0.0)
        self.total.assign(0.0)


def create_model(input_dim, output_dim):

    # --------------------------------------------------
    # 1) Inputs
    # --------------------------------------------------
    inp = tf.keras.layers.Input(shape=(input_dim,), name="model_input")
    feat_inp = tf.keras.layers.Lambda(lambda x: x[:, :-1], name="feature_input")(inp)
    hand_inp = tf.keras.layers.Lambda(lambda x: x[:, -1:], name="handedness_input")(inp)

    # --------------------------------------------------
    # 2) Slice finger/global features
    # --------------------------------------------------
    thumb_feat = tf.keras.layers.Lambda(lambda x: x[:, 0:3], name="thumb_slice")(
        feat_inp
    )
    index_feat = tf.keras.layers.Lambda(lambda x: x[:, 3:6], name="index_slice")(
        feat_inp
    )
    middle_feat = tf.keras.layers.Lambda(lambda x: x[:, 6:9], name="middle_slice")(
        feat_inp
    )
    ring_feat = tf.keras.layers.Lambda(lambda x: x[:, 9:12], name="ring_slice")(
        feat_inp
    )
    pinky_feat = tf.keras.layers.Lambda(lambda x: x[:, 12:15], name="pinky_slice")(
        feat_inp
    )
    # normal_vector_feat = tf.keras.layers.Lambda(
    #     lambda x: x[:, 15:18], name="normal_vector"
    # )(feat_inp)
    # global_feat = tf.keras.layers.Lambda(
    #     lambda x: x[:, 15:input_dim], name="global_slice"
    # )(feat_inp)

    # --------------------------------------------------
    # 3) Handedness concat to each branch input
    # --------------------------------------------------
    thumb_in = tf.keras.layers.Concatenate(name="thumb_input")(
        [thumb_feat, hand_inp]
    )  # 3 + 1
    index_in = tf.keras.layers.Concatenate(name="index_input")(
        [index_feat, hand_inp]
    )  # 3 + 1
    middle_in = tf.keras.layers.Concatenate(name="middle_input")(
        [middle_feat, hand_inp]
    )  # 3 + 1
    ring_in = tf.keras.layers.Concatenate(name="ring_input")(
        [ring_feat, hand_inp]
    )  # 3 + 1
    pinky_in = tf.keras.layers.Concatenate(name="pinky_input")(
        [pinky_feat, hand_inp]
    )  # 3 + 1
    # global_in = tf.keras.layers.Concatenate(name="global_input")(
    #     [global_feat, hand_inp]
    # )  # 3 (Maybe?..) + 1

    l2_reg = tf.keras.regularizers.l2(1e-4)

    # --------------------------------------------------
    # 4) Branch builder
    # --------------------------------------------------
    def finger_branch(x, prefix: str):
        x = tf.keras.layers.Dense(
            8, activation="relu", kernel_regularizer=l2_reg, name=f"{prefix}_dense1"
        )(x)
        x = tf.keras.layers.Dense(4, activation="relu",
                         kernel_regularizer=l2_reg,
                         name=f"{prefix}_dense2")(x)
        out = tf.keras.layers.Dense(
            1, activation="sigmoid", kernel_regularizer=l2_reg, name=f"{prefix}_out"
        )(x)
        return x, out

    thumb, thumb_out = finger_branch(thumb_in, "thumb")
    index, index_out = finger_branch(index_in, "index")
    middle, middle_out = finger_branch(middle_in, "middle")
    ring, ring_out = finger_branch(ring_in, "ring")
    pinky, pinky_out = finger_branch(pinky_in, "pinky")

    # --------------------------------------------------
    # 6) Fusion
    # --------------------------------------------------
    fusion1 = tf.keras.layers.Concatenate(name="fusion1")(
        # [thumb, index, middle, global_branch, hand_inp]
        [thumb, index]
    )
    fusion2 = tf.keras.layers.Concatenate(name="fusion2")(
        # [thumb, index, middle, global_branch, hand_inp]
        [thumb, middle]
    )
    x1 = tf.keras.layers.Dense(
        12, activation="relu", kernel_regularizer=l2_reg, name="fusion_dense11"
    )(fusion1)
    x1 = tf.keras.layers.Dense(
        6, activation="relu", kernel_regularizer=l2_reg, name="fusion_dense12"
    )(x1)

    x2 = tf.keras.layers.Dense(
        12, activation="relu", kernel_regularizer=l2_reg, name="fusion_dense21"
    )(fusion2)
    x2 = tf.keras.layers.Dense(
        6, activation="relu", kernel_regularizer=l2_reg, name="fusion_dense22"
    )(x2)

    out_pinch1 = tf.keras.layers.Dense(1, activation="sigmoid", name="pinch_output_1")(x1)
    out_pinch2 = tf.keras.layers.Dense(1, activation="sigmoid", name="pinch_output_2")(x2)

    out = tf.keras.layers.Concatenate(name="gesture_out")(
        [
            thumb_out,
            index_out,
            middle_out,
            ring_out,
            pinky_out,
            out_pinch1,
            out_pinch2,
        ]
    )

    model = tf.keras.Model(
        inputs=inp,
        outputs=out,
        name="gesture_branch_model",
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.F1Score(average="micro", threshold=0.8, name="f1_score"),
        ],
    )

    model.summary()

    return model


def train_model(model, X_train, y_train, X_val, y_val):

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=100, restore_best_weights=True
        )
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=500,
        batch_size=128,
        callbacks=callbacks,
        verbose=1,
    )

    return history


def run_training(csv_path, DATA_HEADER, LABEL_HEADER):

    X, y = load_dataset(csv_path, DATA_HEADER, LABEL_HEADER)

    X_train, X_val, y_train, y_val, scaler = prepare_data(X, y)

    model = create_model(input_dim=len(DATA_HEADER), output_dim=len(LABEL_HEADER))

    history = train_model(model, X_train, y_train, X_val, y_val)

    model.save("gesture_model.keras")

    with open("gesture_scaler.pkl", "wb") as f:
        pickle.dump(scaler, f)

    return model, scaler, history

# Train

In [245]:
model, scalar, history = run_training(CSV_PATHS, DATA_HEADER, LABEL_HEADER)

           th_open      in_open      mi_open      ri_open      pi_open  \
count  2344.000000  2344.000000  2344.000000  2344.000000  2344.000000   
mean      0.511092     0.142918     0.317833     0.473549     0.473549   
std       0.499984     0.350064     0.465733     0.499406     0.499406   
min       0.000000     0.000000     0.000000     0.000000     0.000000   
25%       0.000000     0.000000     0.000000     0.000000     0.000000   
50%       1.000000     0.000000     0.000000     0.000000     0.000000   
75%       1.000000     0.000000     1.000000     1.000000     1.000000   
max       1.000000     1.000000     1.000000     1.000000     1.000000   

          ti_touch     tm_touch  
count  2344.000000  2344.000000  
mean      0.330631     0.155717  
std       0.470541     0.362664  
min       0.000000     0.000000  
25%       0.000000     0.000000  
50%       0.000000     0.000000  
75%       1.000000     0.000000  
max       1.000000     1.000000  


Model: "gesture_branch_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ model_input         │ (None, 16)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ feature_input       │ (None, 15)        │          0 │ model_input[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ thumb_slice         │ (None, 3)         │          0 │ feature_input[0]… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ handedness_input    │ (None, 1)         │          0 │ model_input[0][0] │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ index_slice         │ (None, 3)         │          0 │ feature_input[0]… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ middle_slice        │ (None, 3)         │          0 │ feature_input[0]… │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ thumb_input         │ (None, 4)         │          0 │ thumb_slice[0][0… │
│ (Concatenate)       │                   │            │ handedness_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ index_input         │ (None, 4)         │          0 │ index_slice[0][0… │
│ (Concatenate)       │                   │            │ handedness_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ middle_input        │ (None, 4)         │          0 │ middle_slice[0][… │
│ (Concatenate)       │                   │            │ handedness_input… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ thumb_dense1        │ (None, 8)         │         40 │ thumb_input[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ index_dense1        │ (None, 8)         │         40 │ index_input[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ middle_dense1       │ (None, 8)         │         40 │ middle_input[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ thumb_dense2        │ (None, 4)         │         36 │ thumb_dense1[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ index_dense2        │ (None, 4)         │         36 │ index_dense1[0][… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ middle_dense2       │ (None, 4)         │         36 │ middle_dense1[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ring_slice (Lambda) │ (None, 3)         │          0 │ feature_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pinky_slice         │ (None, 3)         │          0 │ feature_input[0]… │
│ (Lambda)            │                   │            │                 

 Total params: 791 (3.09 KB)

 Trainable params: 791 (3.09 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - f1_score: 0.0000e+00 - loss: 0.6869 - val_f1_score: 0.0000e+00 - val_loss: 0.6646
Epoch 2/500
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - f1_score: 0.0000e+00 - loss: 0.6504 - val_f1_score: 0.0000e+00 - val_loss: 0.6282
Epoch 3/500
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - f1_score: 0.0000e+00 - loss: 0.6139 - val_f1_score: 0.0000e+00 - val_loss: 0.5932
Epoch 4/500
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - f1_score: 0.0000e+00 - loss: 0.5779 - val_f1_score: 0.0000e+00 - val_loss: 0.5595
Epoch 5/500
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - f1_score: 0.0000e+00 - loss: 0.5433 - val_f1_score: 0.0000e+00 - val_loss: 0.5277
Epoch 6/500
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - f1_score: 0.0013 - loss: 0.5115 - val_f1_score: 0.0180 - val_loss: 0.4974
Epoch 7/500
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - f1_score: 0.1084 - loss: 0.4817 - val_f1_score: 0.1909 - val_loss: 0.4690
Epoch 8/500
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - f1_score: 0.2256 

In [246]:
def predict_gesture(model, scaler, feature):

    x = np.array(feature, dtype=np.float32).reshape(1, -1)

    x = scaler.transform(x)

    pred = model(x).numpy()[0]

    return pred


def predict_hand_gesture_by_dnn(detection_result, model, scalar):

    left_result = None
    right_result = None

    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    if not hand_landmarks_list:
        return {"left": None, "right": None}
    

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        hand_label = handedness_list[idx][0].category_name

        hand_side = -1
        if hand_label == "Right":
            hand_side = 1
        elif hand_label == "Left":
            hand_side = 0
        # hand_side = hand_side * 100

        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

        v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19],:]
        v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],:]

        v = v2 - v1
        v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]

        angle = np.arccos(np.einsum(
            'nt,nt->n',
            v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
            v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]
        ))

        angle = np.degrees(angle)

        # TODO It will be used later... maybe?..
        # v_p1 = joint[[0, 0], :]
        # v_p2 = joint[[5, 17], :]
        # v_palm = v_p2 - v_p1

        # palm_normal = np.cross(v_palm[0], v_palm[1])
        # palm_normal = palm_normal / np.linalg.norm(palm_normal)
        # palm_normal = palm_normal * 500
        # full_data = np.append(angle, palm_normal)
        full_data = angle

        palm_size = np.linalg.norm(joint[17] - joint[5]) + 1e-6

        tdi = np.linalg.norm(joint[4] - joint[8]) / palm_size
        tdm = np.linalg.norm(joint[4] - joint[12]) / palm_size

        # full_data = np.append(full_data, tdi)
        # full_data = np.append(full_data, tdm)
        full_data = np.append(full_data, hand_side)

        data = np.array([full_data], dtype=np.float32)
    
        # print(data)
        # TODO Change model
        # ret, results, neighbours, dist = knn.findNearest(data, 3)
        # print(results)
        # if dist[0][0] < 2000.0:
        pred = predict_gesture(model, scalar, data)
        # label = int(results[0][0])

        if hand_label == "Right":
            right_result = pred
        elif hand_label == "Left":
            left_result = pred

    return {
        "left": left_result,
        "right": right_result
    }


# Result Test

In [247]:
cap = cv2.VideoCapture(CAMERA)
if not cap.isOpened():
    raise RuntimeError(f"웹캠을 열 수 없습니다. camera_index={CAMERA}")

detector = create_hand_detector(model_path="hand_landmarker.task", num_hands=MAX_HANDS)
hand_connections = get_drawing_utils()
# knn = load_knn()
try:
    while True:
        ret, frame = capture_frame(cap)
        if not ret:
            print("프레임을 읽지 못했습니다. 종료합니다.")
            break

        rgb_frame, detection_result = detect_hand_landmarks(detector, frame)
        result = predict_hand_gesture_by_dnn(detection_result, model, scalar)
        annotated_image = draw_landmarks_on_image(rgb_frame, detection_result, hand_connections)

        # OpenCV는 BGR 기준으로 표시하므로 RGB -> BGR로 변환
        display_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

        font = cv2.FONT_HERSHEY_SIMPLEX
        line_height = 26

        pred_left = result["left"]
        if pred_left is None:
            pred_left = [0 for _ in LABEL_HEADER]
        pred_right = result["right"]
        if pred_right is None:
            pred_right = [0 for _ in LABEL_HEADER]

        # ===== LEFT HAND =====
        lines_left = ["LEFT HAND"]

        for name, p in zip(LABEL_HEADER, pred_left):
            lines_left.append(f"{name:<10} : {p:>6.2%}")

        for i, line in enumerate(lines_left):
            y = 60 + i * line_height
            cv2.putText(
                display_image,
                line,
                (10, y),
                font,
                0.6,
                (0,255,0),
                2,
                cv2.LINE_AA
            )


        # ===== RIGHT HAND =====
        lines_right = ["RIGHT HAND"]

        for name, p in zip(LABEL_HEADER, pred_right):
            lines_right.append(f"{name:<10} : {p:>6.2%}")

        for i, line in enumerate(lines_right):
            y = 60 + i * line_height
            cv2.putText(
                display_image,
                line,
                (350, y),
                font,
                0.6,
                (255,255,0),
                2,
                cv2.LINE_AA
            )


        cv2.imshow("MediaPipe Hand Landmarks", display_image)

        # ESC 또는 q 입력 시 종료
        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord("q"):
            break
        
finally:
    cap.release()
    cv2.waitKey(-1)
    cv2.destroyAllWindows()